# 도구

* 도구는 AI 에이전트가 추가 정보와 컨텍스트를 검색하고 작업을 수행하며 환경과 의미 있게 상호작용하도록 함
* AI에서의 도구는 에이전트가 원하는 결과를 달성하기 위해 수행할 수 있는 특정 기능 또는 일련의 작업을 말함

* AI 에이전트는 환경과 상호작용하고 정보를 처리하며 작업을 자율적으로 실행하도록 설계된 정교한 시스템  
-> 이를 효율적으로 수행하려면 체계회된 도구 모음이 필요함

## 랭체인 기초

* 랭체인의 중심에는 프롬프트를 처리하고 응답을 생성하는 파운데이션 모델을 감싼 채팅 모델이 있음
* init_chat_model()에 모델 이름을 파라미터로 전달해 다양한 LLM을 동일한 채팅 모델 인터페이스로 초기화 가능

In [ ]:
from langchain.chat_models import init_chat_model
llm = init_chat_model(model='gpt-5-mini', temperature=0)

* 랭체인은 대화 컨텍스트를 유지하기 위해 상호작용을 메시지로 구조화
    * HumanMessage: 사용자 입력
    * AIMessage: 모델의 응답
    * ToolMessage: 도구 호출 결과

In [6]:
from langchain_core.messages import HumanMessage, ToolMessage
messages = [HumanMessage("오늘 날씨가 어떤가요?")]

* 도구는 모델이 텍스트 생성 능력을 넘어 기능읗 확장하도록 호출할 수 있는 외부 함수
    * API 호출, DB 조회, 계산 수행 등
* @tool 데코레이터를 사용해 도구를 정의
    * 함수를 등록하고 출력 스키마를 자동으로 생성

In [3]:
from langchain_core.tools import tool

@tool
def add(x: float, y: float) -> float:
    """'x'와 'y'를 더합니다."""
    return x + y

* 도구를 정의한 뒤 .bind_tools()로 모델에 바인딩  
-> 모델이 사용자 입력에 응답하는 과정에서 해당 도구를 선택해 호출할 수 있음
* 모델과 상호작용 시 현재 대화를 나타내는 메시지 목록을 .invoke()에 전달
* 모델이 도구 호출을 결정하면 도구 호출이 출력에 포함
* 해당 함수를 실행해 결과를 대화에 추가하고 최종 응답 생성을 이어감

In [ ]:
llm_with_tools = llm.bind_tools([add])
ai_msg = llm_with_tools.invoke(messages)
for tool_call in ai_msg.tool_calls:
    tool_response = add_numbers.invoke(tool_call)

### 로컬 도구

* 로컬에서 실행하는 도구
* 보통 특정 작업에 맞춘 사전 정의 규칙과 로직을 따라 만듦
* 만들고 수정하기 쉽고 에이전트와 함께 배포
* 정확성, 예측 가능성, 단순성을 제공
* 로직이 명시적으로 정의되므로 대체로 예측 가능하고 신뢰할 수 있음

* 메타데이터 또한 중요함
* 모델은 해당 메타데이터를 바탕으로 어떤 도구를 호출할지 결정

* 중요한 과정
    * 범위를 좁게 잡고 정확한 이름을 선택
    * 명확하고 구별되는 설명을 작성
    * 엄격한 입력/출력 스키마를 정의

* 중요한 단점
    * 확장성
        * 로컬 도구의 설계, 구축, 배포는 번거롭고 사용 사례 전반에 걸쳐 공유하기가 더 어려움
    * 중복
        * 로컬 도구를 사용하려는 팀이나 모든 팀이나 에이전트 배포는 해당 에이전트 서비스와 함께 같은 라이브러리를 배포해야 함
        * 변경 사항 배포 시 각 에이전트 서비스의 배포를 조율해야 함
    * 유지보수
        * 환경이나 요구사항이 바뀌면 수작업으로 만든 도구는 업데이트와 조정이 자주 필요할 수 있음
        * 리소스를 많이 소모하며 일반적으로 에이전트 서비스의 재배포가 필요함

* 수작업으로 만든 도구는 파운데이션 모델이 가진 약점을 해결하는 데 특히 유용함
* 계산기 도구를 사용하는 예제

In [ ]:
from langchain_core.runnables import ConfigurableField
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

# 간결한 함수 정의로 도구를 정의
@tool
def multiply(x: float, y: float) -> float:
    """'x'에 'y'를 곱합니다."""
    return x * y

@tool
def exponentiate(x: float, y: float) -> float:
    """'x'를 'y' 제곱합니다."""
    return x ** y

@tool
def add(x: float, y: float) -> float:
    """'x'와 'y'를 더합니다."""
    return x + y

tools = [multiply, exponentiate, add]

# LLM 초기화 및 도구 바인딩
llm = init_chat_model(model="gpt-5-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)

* 바인딩 작업은 도구를 등록함
* 내부적으로 랭체인은 파운데이션 모델의 응답에 도구 호출 요청이 포함되는지 확인
* 파운데이션 모델에 질문을 던지면 질문에 답하는 데 도움이 된다면 도구를 선택하고 해당 도구의 파라미터를 정해 함수를 호출함

In [ ]:
query = "393 * 12.25는 얼마인가요? 그리고 11 + 49는요?"
messages = [HumanMessage(query)]

ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    selected_tool = {
        "add": add,
        "multiply": multiply,
        "exponentiate": exponentiate
    }[tool_call["name"]]
    result = selected_tool.invoke(tool_call['args'])

    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    print(f"Result: {result}")

    # ToolMessage 생성
    tool_msg = ToolMessage(content=str(result), tool_call_id=tool_call["id"])
    messages.append(tool_msg)

final_response = llm_with_tools.invoke(messages)
print(final_response.content)

* 임의로 유용하고 영향력 있는 프로그램을 파운데이션 모델에 바인딩할 수 있으며 어떤 프로그램을 어떤 파라미터로 실행할지 파운데이션 모델에 맡길 수 있음

### API 기반 도구

* API 기반 도구는 자율 에이전트가 외부 서비스와 상호작용하도록 하여 로컬에서 수행하기 어려운 데이터 처리나 액션 실행을 가능하게 기능을 확장함
* API 기반 도구는 에이전트가 여러 외부 시스템과 통합하거나 실시간 데이터를 가져오거나 내부에서 처리하기엔 자원 소모가 큰 복잡한 연산을 수행해야 하는 상황에서 특히 유용함
* 외부 서비스를 활용하면 에이전트가 에이전트가 수행할 수 있는 작업의 범위가 크게 넓어짐
* API 기반 도구의 또다른 장점은 실시간 데이터 접근 기능

* API 기반 도구 구현 예제

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.messages import HumanMessage

api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=300)
tool = WikipediaQueryRun(api_wrapper=api_wrapper)

# LLM 초기화 및 도구 바인딩
llm = init_chat_model(model="gpt-5-mini", temperature=0)
llm_with_tools = llm.bind_tools([tool])

messages = [HumanMessage("Buzz Aldrin의 주요 업적은 무엇인가요?")]

ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_msg = tool.invoke(tool_call)

    print(tool_msg.name)
    print(tool_call['args'])
    print(tool_msg.content)
    messages.append(tool_msg)
    print()

final_response = llm_with_tools.invoke(messages)
print(final_response.content)

* 주식 시장 데이터를 조회하고 표시하는 에이전트 예제

In [ ]:
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
import requests

@tool
def get_stock_price(ticker: str) -> float:
    """주식 시장 거래소 거래 티커에 대한 주식 가격을 가져옵니다."""
    api_url = f"https://api.example.com/stocks/{ticker}"
    try:
        response = requests.get(api_url)
        if response.status_code == 200:
            return response.json()["price"]
        else:
            return f"주식 가격을 가져오는데 실패했습니다: {ticker}"
    except requests.exceptions.RequestException:
        return f"주식 가격을 가져오는데 실패했습니다: {ticker}"

# LLM 초기화 및 도구 바인딩
llm = init_chat_model(model="gpt-5-mini", temperature=0)
llm_with_tools = llm.bind_tools([get_stock_price])

messages = [HumanMessage("애플의 현재 주식 가격은 얼마인가요?")]

ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_msg = get_stock_price.invoke(tool_call)

    print(tool_msg.name)
    print(tool_call['args'])
    print(tool_msg.content)
    messages.append(tool_msg)
    print()

final_response = llm_with_tools.invoke(messages)
print(final_response.content)

* 에이전트가 작업 수행에 필요한 정보에 접근하고 이를 처리할 수 있는 전용 도구를 제공하면 자동화할 수 있는 작업의 범위와 복잡도를 크게 확장할 수 있음
* API 도구를 설계할 때는 신뢰성, 보안성, 우아한 실패에 중점을 두어야 함
* API의 요청 제한을 주의해 서비스 중단을 방지하고 개인정보 보호법을 준수하기 위해 사용자 데이터를 익명화하거나 마스킹해야 함
* 네트워크 문제나 잘못된 응답으로 인한 오류를 견고하게 처리해 사용자 경험이 손상되지 않도록 해야 함

### 플러그인 도구

* 플러그인 도구는 모듈화되어 있으며 최소한의 커스터마이징만으로 AI 에이전트 프레임워크에 통합할 수 있는 구성 요소
* 이런 도구는 기존 라이브러리, API, 서드파티 서비스를 활용해 광범위한 개발 작업 없이도 에이전트의 기능을 확장함
* 플러그인 도구는 에이전트의 기능을 빠르게 배포하고 확장할 수 있도록 해주며 사전에 설계된 모듈로서 적은 노력으로 AI 시스템에 통합할 수 있음

* 플러그인 도구의 한 가지 장점은 모델 실행 계층에서의 통합 가능성
* 이러한 도구를 기존 워크플로에 거의 영향을 주지 않고 AI 모델에 추가할 수 있음
* 다만 플러그인 도구는 강력하지만 로컬 또는 원격에서 제공되는 맞춤형 툴처럼 높은 수준의 커스터마이징과 적응성을 제공하지 않음

* 주요 플랫폼이 제공하는 플러그인 도구 카탈로그는 빠르게 확장됨
* 오픈소스 파운데이션 모델에서도 빠르게 성장하는 플러그인 도구 생태계가 형성되고 있음
* 플러그인 도구의 실제 응용 범위는 매우 넓으며 산업과 사용 사례 전반에 걸쳐 다양함

### MCP

* 각 데이터 소스나 서비스별로 커스텀 어댑터를 작성하는 통합 방식은 유지보수가 어렵고 확장성이 낮음  
-> 모델 컨텍스트 프로토콜(MCP)

* MCP는 앤트로픽이 처음 제안한 표준
* 모델에 독립적인 통합 표준을 제공해 LLM과 외부 시스템을 연결하는 통일된 방식의 인터페이스 역할을 함

* MCP의 두 가지 주요 구성 요소
    * MCP 서버
        * 표준화된 JSON-RPC 2.0 인터페이스를 통해 데이터나 서비스를 노출하는 웹 서버
        * 클라우드 오브젝트 스토리지, SQL DB, CRM, 내부 비즈니스 로직 등 MCP 사양을 구현하기만 하면 어떤 시스템이든 서버로 래핑 가능
    * MCP 클라이언트
        * MCP를 사용하는 에이전트 또는 LLM 애플리케이션
        * JSON-RPC 요청을 전송하고 구조화된 JSON 응답을 받음
        * 프로토콜이 일관되므로 개발자는 서버의 내부 작동 원리를 몰라도 노출된 메서드만 알면 됨

* MCP는 내부적으로 HTTPS 또는 WebSocket에서 JSON-RPC 2.0을 사용
* 서버는 자신이 제공하는 메서드와 입출력 스키마를 공개
* 클라이언트는 메서드 카탈로그를 조회해 어떤 메서드를 어떤 파라미터로 호출할지 추론
* 도구 호출이 결정되면 MCP 클라이언트는 해당 호출을 JSON-RPC 페이로드로 래핑해 서버로 전송하고 응답을 기다림

* 이점이 많으나 여러 보안 이슈가 제기됨
    * MCP 엔드포인트를 공유하는 경우의 인증, 접근 제어, 잠재적인 공격 벡터와 관련된 문제

* MCP를 실제로 사용하는 예제
    1. 로컬에서 'math' MCP 서버를 서브프로세스로 실행
    2. localhost:8000/mcp에서 실행중인 원격 'weather' MCP 서버에 연결
    3. 사용자는 마지막 메시지를 검사해 산술 표현식이면'math' 도구를, 날씨 질의면 'weather' 도구를 호출하는 비동기 에이전트 루프를 구현
    4. 에이전트가 도구의 출력을 파싱하고 최종 assistant 응답을 반환하는 방식을 보여줌

In [ ]:
import asyncio
from typing import Any, Sequence, TypedDict

from langchain_core.messages import HumanMessage
from langchain_core.tools import Tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph

class AgentState(TypedDict):
    messages = Sequence[Any]

# MCP 도구 클라이언트 생성
mcp_client = MultiServerMCPClient(
    {
        "math": {
            "command": "python3",
            "args": ["ch04/mcp_servers/MCP_math_server.py"],
            "transport": "stdio",
        },
        "weather": {
            # `python ch04/mcp_servers/MCP_weather_server.py`으로 MCP 서버를 먼저 실행
            "url": "http://0.0.0.0:8000/mcp",
            "transport": "streamable_http",
        },
    }
)

# MCP 도구 클라이언트에서 도구 목록을 가져옴
async def get_mcp_tools() -> list[Tool]:
    return await mcp_client.get_tools()

async def call_mcp_tools(state: AgentState) -> dict[str, Any]:
    messages = state["messages"]
    last_msg = messages[-1].content.lower()

    # MCP_TOOLS를 전역 변수로 선언해 한 번만 가져옴
    global MCP_TOOLS
    if "MCP_TOOLS" not in globals():
        MCP_TOOLS = await mcp_client.get_tools()

    # 간단한 판단: 숫자/연산자 토큰이 있으면 "math"
    # 'weather'나 '날씨'가 포함되어 있으면 "weather" 선택
    # 실제로는 LLM이 적절한 도구를 판단
    if any(token in last_msg for token in ["+", "-", "*", "/", "(", ")"]):
        tool_name = "math"
        tool_input = {"expression": messages[-1].content}
    elif "weather" in last_msg or "날씨" in messages[-1].content:
        tool_name = "weather"
        content = messages[-1].content

        if "weather in" in content.lower():
            location = content.lower().split("weather in")[1].strip().rstrip("?").strip()
        elif "의 날씨" in content:
            location = content.split("의 날씨")[0].strip()
        else:
            location = "NYC"
        tool_input = {"location": location}
    else:
        return {
            "messages": [
                {
                    "role": "assistant",
                    "content": "수학 또는 날씨 질문만 답변할 수 있습니다."
                }
            ]
        }

    tool_obj = next((t for t in MCP_TOOLS if t.name == tool_name), None)
    if tool_obj is None:
        return {
            "messages": [
                {
                    "role": "assistant",
                    "content": f"{tool_name} 도구를 사용할 수 없습니다."
                }
            ]
        }

    mcp_result : str = await tool_obj.ainvoke(tool_input)

    return {
        "messages": [
            {"role": "assistant", "content": mcp_result}
        ]
    }

def construct_graph():
    g = StateGraph(AgentState)
    g.add_node("assistant", call_mcp_tools)
    g.set_entry_point("assistant")
    return g.compile()

GRAPH = construct_graph()

async def run_math_query():
    initial_state = {
        "messages": [
            HumanMessage(content="(3 + 5) * 12는 얼마인가요?")
        ]
    }
    result = await GRAPH.ainvoke(initial_state)
    assistant_msg = result["messages"][-1]
    content = assistant_msg.get("content") if isinstance(assistant_msg, dict) else assistant_msg.content
    print("Math answer:", content)

async def run_weather_query():
    initial_state = {
        "messages": [
            HumanMessage(content="NYC의 날씨는 어때요?")
        ]
    }
    result = await GRAPH.ainvoke(initial_state)
    assistant_msg = result["messages"][-1]
    print("Weather answer:", assistant_msg["cotent"])


if __name__ == "__main__":
    asyncio.run(run_math_query())
    asyncio.run(run_weather_query())

* 'math' 도구 호출 시 command와 args를 사용해 MCP_weather_server.py를 실행하는 서브프로세스 생성
* 'weather' 도구 호출 시 이미 실행 중인 HTTP MCP 서버를 가리킴

* MCP는 서비스 구현을 에이전트 로직에서 분리함
* 여러 에이전트가 별도의 커스텀 통합 없이 동일한 도구를 재사용할 수 있음

### 상태 유지 도구

* 상태 유지 도구는 로컬 스크립트, 외부 API, MCP에서 배포된 서비스 모두에 존재할 수 있지만 같은 약점을 안고 있음
    * 파운데이션 모델에 지속적인 상태를 직접 조작할 권한을 부여하면 모델이 파괴적인 실수를 하거나 악성 사용자가 악용할 수 있음
* 위험을 줄이기 위해서 범위가 좁게 정의된 작업만 도구로 등록해야 함
* 도구 등록 단계에서 기능을 제한하면 공격 표면을 줄이고 잠재적 오류의 범위를 명확히 제한할 수 있음

=> 궁극적으로 최소 권한 원칙이 설계의 기준이 되어야 함

## 도구 개발 자동화

* 코드 생성은 AI 역량의 혁신적인 도약을 의미하며 특히 에이전트가 실시간으로 자신의 도구를 작성해 작업을 수행하거나 새로운 API와 상호작용할 때 그 잠재력이 두드러짐
* AI 에이전트가 기능을 스스로 확장하고 적응할 수 있도록 만들어 유연성과 문제 해결 능력을 크게 강화함

### 파운데이션 모델을 활용한 도구 개발

* 파운데이션 모델은 이제 직접 도구를 생성함
* LLM에 API ㅁ여세나 샘플 입력값을 제공하면 초기 래퍼, 헬퍼 함수, 상위 수준의 원자적 연산을 자동으로 생성할 수 있음
* 모델이 임시로 코드를 작성해 안전한 샌드박스 환경에서 실행한 뒤 스스로 결과를 평가할 수 있음
* 빠른 반복 과정을 거치면 수작업으로 일일이 래퍼를 만들지 않아도 에이전트가 직접 호출할 수 있는 검증된 소규모 도구 집합을 얻게 됨

* 복잡한 API 환경을 다룰 때 유용함
* 오픈API 명세서나 코드 샘플을 제공하면 각 함수의 초안을 자동으로 생성
* 이후 사람이 생성된 코드를 검토하고 보안성과 정확성을 확보한 뒤 CI/CD 파이프라인에 포함시킴
* API가 변경될 때마다 동일한 루프를 다시 실행하면 도구를 최신 상태로 유지할 수 있음

* 다만 명확한 검증 기준과 개발자의 감독이 필수적
* 최종적으로 엣지 케이스를 점검하고 보안 취약점을 차단하며 비즈니스 로직이 올바르게 반영되었는지 확인하는 책임은 개발자에게 있음

### 실시간 코드 생성
* AI 에이전트가 작동 중 필요에 따라 코드를 작성하고 실행하는 기능
* 에이전트는 특정 작업을 수행하기 위해 새로운 도구를 생성하거나 기존 도구를 수정할 수 있으며 그 결과 매우 높은 적응력을 갖게 됨

* 이 과정은 에이전트가 현재 작업을 분석하고 이를 수행하기 위한 단계를 결정하는 것부터 시작
* 에이전트는 이해한 내용을 바탕으로 코드 스니펫을 작성하고 이를 실행
* 코드가 기대한 대로 작동하지 않으면 반복적으로 수정하며 각 시도에서 학습해 원하는 결과에 도달할 때까지 개선  
-> 에이전트가 도구를 지속적으로 정제하고 성능을 향상시키며 자율적으로 확장할 수 있게 함

* 적응성과 효율성 측면에서 강력한 이점을 제공
* 새로운 작업과 환경에 신속하게 대응할 수 있음  
-> 다운타임을 줄이며 전체 효율성을 크게 향상

* 품질 관리의 문제가 가장 큼
* 자율적으로 생성된 코드의 품질과 보안을 보장하지 못하면 시스템 오류, 보안 침해 등 심각한 문제가 발생할 수 있음
* 자체 생성한 코드를 실행할 수 있도록 허용할 경우 악의적인 행위자가 이를 악용해 악성 코드를 삽입할 위험이 존재  
-> 견고한 보안 조치와 감독 체계를 반드시 마련해야 함

* 재현성 문제도 중요한 단점
* 에이전트가 매번 도구를 처음부터 다시 생성하면 예측 가능성을 잃게 됨
* 한 번의 호출에서 성공했다고 해서 다음 호출에서도 동일한 결과를 보장할 수 없음  
-> 불안정성은 에이전트의 일관된 작동을 보증하기 어렵게 함

* 리소스 소비 역시 중요한 고려사항  
-> 시스템 성능의 여러 측면에 걸쳐 가드레일을 설정해 이러한 위험을 완화해야 함 

## 도구 사용 설정

*  파운데이션 모델 API는 파라미터 tool-choice를 통해 모델의 도구 사용 방식을 명시적으로 제어할 수 있음  
-> 이를 통해 유연한 호출 방식에서 결정적 기능으로 전환할 수 있음

* auto 모드
    * 모델이 컨텍스트에 따라 도구 호출 여부를 스스로 결정하며 일반적인 용도에 적합
* any or required 모드
    * 모델이 최소 한 개의 도구를 반드시 호출하도록 강제하므로 도구의 출력이 필수적인 경우에 이상적
* none
    * 모든 도구 호출이 차단, 제어된 출력이나 테스트 혼경에서 활용 가능
* 일부 인터페이스는 특정 도구를 고정해 예측 가능하고 반복 가능한 흐름을 보장하기도 함

* 에이전트가 도구 호출을 건너뛰거나 잘못된 JSON을 생성하거나 오류가 발생한 도구를 실행하는 등 예상치 못한 오작동을 보일 수 있으므로 신뢰할 수 있는 폴백 메커니즘과 후처리 체계를 반드시 마련해야 함

* 모델의 각 응답 후에는 다음 사항을 점검해야 함
    * 모델이 올바른 도구를 호출했는지
    * 유효한 JSON을 생성했는지
    * 런타임 오류 없이 성공적으로 실행되었는지

* 문제 발견 시 교정 절차
    * 스키마 검증
        * jsonschema나 Pydantic을 사용해 누락된 필드나 잘못된 구조를 탐지
        * 도구 호출이 누락된 경우 자동으로 실행하고 JSON이 유효하지않다면 모델에 수정 프롬프트를 보냄
    * 지능적 재시도
        * 일시적 오류에는 지수 백오프같은 구조화된 재시도 로직을 적용하거나 전체 과정을 다시 시작하지 않고 문제된 부분만 재생성
    * 안정적인 폴백
        * 재시도가 실패하면 백업 모델이나 서비스를 사용하거나 사용자에게 추가 입력을 요청하거나 캐시된 데이터를 활용하거나 안전한 기본값을 반환
    * 로깅
        * 프롬프트, 도구 호출, 검증 오류, 재시도, 폴백 등 모든 단계를 기록해 관측 가능성을 확보하고 디버깅과 개선에 활용